In [22]:
from pathlib import Path
import os
import random
import shutil
import pandas as pd

dataset_path = Path("aptos2019-blindness-detection")
train_csv = dataset_path / "train.csv"
test_csv = dataset_path / "test.csv"
train_images_dir = dataset_path / "train_images"
test_images_dir = dataset_path / "test_images"

output_root = dataset_path / "imagefolder"
train_out_dir = output_root / "train"
val_out_dir = output_root / "val"
test_out_dir = output_root / "test"

CLASS_LABELS = {
    0: "No DR",
    1: "Mild",
    2: "Moderate",
    3: "Severe",
    4: "Proliferative DR",
}

# Keep originals by default. Set USE_MOVE=True only if you really want to move files.
VAL_RATIO = 0.2
SEED = 42
USE_MOVE = False
CLEAR_OUTPUT = False

required_paths = [train_csv, train_images_dir, test_csv, test_images_dir]
missing_paths = [str(p) for p in required_paths if not p.exists()]
if missing_paths:
    raise FileNotFoundError(f"Missing required dataset paths: {missing_paths}")

if CLEAR_OUTPUT and output_root.exists():
    shutil.rmtree(output_root)

for split_dir in [train_out_dir, val_out_dir]:
    for cls in CLASS_LABELS:
        (split_dir / str(cls)).mkdir(parents=True, exist_ok=True)
test_out_dir.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(train_csv)
required_cols = {"id_code", "diagnosis"}
if not required_cols.issubset(df.columns):
    raise ValueError(f"{train_csv} must contain columns: {required_cols}")

df = df[["id_code", "diagnosis"]].dropna().copy()
df["id_code"] = df["id_code"].astype(str)
df["diagnosis"] = df["diagnosis"].astype(int)

if not set(df["diagnosis"].unique()).issubset(set(CLASS_LABELS.keys())):
    raise ValueError(f"Diagnosis labels must be in {sorted(CLASS_LABELS.keys())}.")

def stratified_train_val_split(dataframe, val_ratio=0.2, seed=42):
    rng = random.Random(seed)
    train_parts = []
    val_parts = []

    for _, class_df in dataframe.groupby("diagnosis"):
        idx = class_df.index.tolist()
        rng.shuffle(idx)

        n_val = max(1, int(round(len(idx) * val_ratio)))
        n_val = min(n_val, max(1, len(idx) - 1)) if len(idx) > 1 else 1

        val_idx = idx[:n_val]
        train_idx = idx[n_val:]

        val_parts.append(dataframe.loc[val_idx])
        if train_idx:
            train_parts.append(dataframe.loc[train_idx])

    train_df = pd.concat(train_parts, ignore_index=True) if train_parts else pd.DataFrame(columns=dataframe.columns)
    val_df = pd.concat(val_parts, ignore_index=True)
    return train_df, val_df

def transfer_file(src: Path, dst: Path, use_move=False):
    if dst.exists():
        return
    if use_move:
        shutil.move(str(src), str(dst))
        return
    try:
        os.link(src, dst)
    except OSError:
        shutil.copy2(src, dst)

def distribute_labeled_images(split_df, split_name):
    written = 0
    missing = []

    for row in split_df.itertuples(index=False):
        src = train_images_dir / f"{row.id_code}.png"
        dst = output_root / split_name / str(int(row.diagnosis)) / f"{row.id_code}.png"

        if not src.exists():
            missing.append(src.name)
            continue

        transfer_file(src, dst, use_move=USE_MOVE)
        written += 1

    return written, missing

train_split_df, val_split_df = stratified_train_val_split(df, val_ratio=VAL_RATIO, seed=SEED)
train_written, train_missing = distribute_labeled_images(train_split_df, "train")
val_written, val_missing = distribute_labeled_images(val_split_df, "val")

test_written = 0
for src in test_images_dir.glob("*.png"):
    dst = test_out_dir / src.name
    if not dst.exists():
        transfer_file(src, dst, use_move=False)
        test_written += 1

split_csv_dir = output_root / "splits"
split_csv_dir.mkdir(parents=True, exist_ok=True)
train_split_df.to_csv(split_csv_dir / "train_split.csv", index=False)
val_split_df.to_csv(split_csv_dir / "val_split.csv", index=False)

mapping_path = output_root / "class_label_mapping.csv"
pd.DataFrame(
    [{"class_id": class_id, "class_name": class_name} for class_id, class_name in CLASS_LABELS.items()]
).to_csv(mapping_path, index=False)

print("Class label mapping:")
for class_id, class_name in CLASS_LABELS.items():
    print(f"  {class_id}: {class_name}")
print()
print(f"Output root: {output_root}")
print(f"Train images placed: {train_written}, missing: {len(train_missing)}")
print(f"Val images placed: {val_written}, missing: {len(val_missing)}")
print(f"Test images placed: {test_written}")
print(f"Saved split CSVs in: {split_csv_dir}")
print(f"Saved label mapping CSV: {mapping_path}")

Class label mapping:
  0: No DR
  1: Mild
  2: Moderate
  3: Severe
  4: Proliferative DR

Output root: aptos2019-blindness-detection/imagefolder
Train images placed: 2929, missing: 0
Val images placed: 733, missing: 0
Test images placed: 1928
Saved split CSVs in: aptos2019-blindness-detection/imagefolder/splits
Saved label mapping CSV: aptos2019-blindness-detection/imagefolder/class_label_mapping.csv


In [23]:
from pathlib import Path
import pandas as pd

dataset_path = Path("aptos2019-blindness-detection")
output_root = dataset_path / "imagefolder"
mapping_path = output_root / "class_label_mapping.csv"

CLASS_LABELS = {
    0: "No DR",
    1: "Mild",
    2: "Moderate",
    3: "Severe",
    4: "Proliferative DR",
}

def count_split(split_name):
    rows = []
    split_dir = output_root / split_name
    for class_id, class_name in CLASS_LABELS.items():
        class_dir = split_dir / str(class_id)
        rows.append({
            "split": split_name,
            "class_id": class_id,
            "class_name": class_name,
            "count": len(list(class_dir.glob("*.png"))) if class_dir.exists() else 0,
        })
    return rows

rows = count_split("train") + count_split("val")
summary_df = pd.DataFrame(rows).sort_values(["class_id", "split"])
pivot_df = (
    summary_df.pivot(index=["class_id", "class_name"], columns="split", values="count")
    .fillna(0)
    .astype(int)
)

print("Class distribution after organization:")
print(pivot_df)
print()
if mapping_path.exists():
    print(f"Label mapping CSV: {mapping_path}")
else:
    print("Label mapping CSV not found yet. Run Cell 1 first.")
print(f"Test images in imagefolder/test/images: {len(list((output_root / 'test' / 'images').glob('*.png')))}")
print(f"Train split CSV: {output_root / 'splits' / 'train_split.csv'}")
print(f"Val split CSV: {output_root / 'splits' / 'val_split.csv'}")

Class distribution after organization:
split                      train  val
class_id class_name                  
0        No DR              1444  361
1        Mild                296   74
2        Moderate            799  200
3        Severe              154   39
4        Proliferative DR    236   59

Label mapping CSV: aptos2019-blindness-detection/imagefolder/class_label_mapping.csv
Test images in imagefolder/test/images: 0
Train split CSV: aptos2019-blindness-detection/imagefolder/splits/train_split.csv
Val split CSV: aptos2019-blindness-detection/imagefolder/splits/val_split.csv
